# OFC conversion constants: DZ(k=1, j=4) to v-mode 1 and to camera defocus (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-09-18  
**Last Modified:** 2026-09-18  
**Status:** Complete  
**Keywords:** AOS, OFC, sensitivity matrix, SVD, double Zernike, v-mode, defocus, hexapod

## Description

Prints the conversion constants between the Double Zernike (DZ) defocus term
**DZ(k=1, j=4)** — focal (field) Zernike order k=1, the field-constant component, of pupil
Noll index j=4 — and the two quantities it is usually compared against: **v-mode 1** of the
Active Optics System (AOS) sensitivity matrix, and an **equivalent hexapod defocus travel**
in µm.

The question has one honest answer in each direction and they are *not* reciprocals of each
other, because the forward map from hexapod motion to wavefront is not square: two hexapod
defocus axes (camera and M2) produce nearly the same wavefront, so inverting the map requires
choosing how to split the motion between them. This notebook states both conventions
explicitly rather than quoting one number.

Key functionality:
1. Builds the sensitivity-matrix singular value decomposition (SVD) through
   `ofc_svd.build_ofc_svd`, the sanctioned engine, and reads the pupil Noll index set from a
   `param_set`'s `visits.parquet` rather than hardcoding it.
2. **Forward:** the wavefront produced per µm of each hexapod defocus axis, from the raw
   sensitivity matrix.
3. **Inverse, SVD minimum-norm:** the degrees of freedom (DOF) and v-modes recovered from a
   unit DZ(k=1, j=4), which is what `fam_dz` and `optical_state` both use.
4. Cross-checks: the round trip through the forward matrix, and the recovered v-mode 1
   against `run_science_lut_analysis.v1_per_um_dz_value`, the constant the focus studies use.

**Output:** printed constants only; no files are written.

**Based on:** `aos/docs/studies/smatrix_vmode.md`, `aos/docs/studies/fam_focus.md`,
`aos/code/aos_state.py`, and `aos/code/science_lut/run_science_lut_analysis.py`
(`v1_per_um_dz_value`).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-09-18 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [The SVD](#data)
5. [Forward: hexapod defocus to wavefront](#forward)
6. [Inverse: DZ(k=1, j=4) to v-modes and DOF](#inverse)
7. [Cross-checks](#checks)
8. [The constants, collected](#results)

<a id='params'></a>
## Parameters

In [1]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# The DZ term of interest: focal (field) Zernike order k and pupil Noll index j.
K_FOCAL = 1                  # k=1 is the field-constant component
J_PUPIL = 4                  # Noll 4, defocus

# SVD scheme. 50 DOF and 34 retained v-modes is the topic-wide default.
K_MIN, K_MAX = 1, 6          # focal orders spanned, matching prefix 'z1toz6'
N_DOF = 50
N_MODES = 34

# The param_set whose visits.parquet supplies the canonical pupil Noll index set. Any
# param_set built with the same donut extraction gives the same set; it is read rather than
# hardcoded because a contiguous range(4, 23) would carry Noll 20 and 21 as NaN and drop 23-26.
PARAM_SET = 'fam_danish_1_2_0_wep17_6_1_refitWCS_bin2x'

# The two hexapod defocus degrees of freedom, by index in the 50-DOF vector.
DOF_M2_DZ = 0                # M2 hexapod dz [um]
DOF_CAM_DZ = 5               # camera hexapod dz [um]

<a id='setup'></a>
## Setup & Imports

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# A notebook has no __file__, so walk up to the topic directory; this works at any depth.
_TOPIC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'code').is_dir())
sys.path.insert(0, str(_TOPIC.parent))        # repo root -> common/
sys.path.insert(0, str(_TOPIC / 'code'))      # flat cross-study modules
for _d in sorted((_TOPIC / 'code').glob('*/')):
    if _d.is_dir() and not _d.name.startswith(('_', '.')):
        sys.path.insert(0, str(_d))

from lsst.ts.intrinsic.wavefront import ofc_svd as osv
from lsst.ts.ofc import OFCData

import run_science_lut_analysis as A

print(f'topic directory: {_TOPIC}')

topic directory: /sdf/home/r/roodman/notebooks/rubin-work/aos


<a id='functions'></a>
## Helper Functions

In [3]:
def read_pupil_j(param_set, topic=None):
    """Canonical pupil Noll indices from a param_set's visits sidecar.

    Parameters
    ----------
    param_set : `str`
        Butler collection paired with a processing variant.
    topic : `pathlib.Path`, optional
        The ``aos`` topic directory; defaults to the one found at import.

    Returns
    -------
    pupil_j : `list` [`int`]
        The pupil Noll indices the donut fit used, dimensionless indices.

    Notes
    -----
    The canonical set is the 21 indices 4-19 and 22-26; Noll 20 and 21 are absent by design.
    Hardcoding a contiguous range would silently carry those two as all-NaN columns and drop
    j=23-26, which is why the sidecar is the only source used.
    """
    path = (topic or _TOPIC) / 'output' / param_set / 'visits.parquet'
    v = pd.read_parquet(path, columns=['nollIndices'])
    return [int(j) for j in np.asarray(v['nollIndices'].iloc[0]).tolist()]


def unit_dz_vector(svd, k, j):
    """A DZ coefficient vector holding 1 µm of wavefront in one (k, j) term and zero elsewhere.

    Parameters
    ----------
    svd : `OFCSvd`
        A `ofc_svd.build_ofc_svd` result.
    k : `int`
        Focal (field) Zernike order.
    j : `int`
        Pupil Noll index.

    Returns
    -------
    W : `numpy.ndarray`
        Shape (1, n_kj) in ``svd.kj_grid`` order [µm of wavefront].
    idx : `int`
        Position of ``(k, j)`` in ``svd.kj_grid``.

    Raises
    ------
    ValueError
        If ``(k, j)`` is outside the SVD's focal-order range or pupil Noll set.
    """
    try:
        idx = svd.kj_grid.index((int(k), int(j)))
    except ValueError:
        raise ValueError(f'(k={k}, j={j}) is not in this SVD: focal orders '
                         f'{svd.k_min}..{svd.k_max}, pupil Noll {svd.iZs}') from None
    W = np.zeros((1, len(svd.kj_grid)))
    W[0, idx] = 1.0
    return W, idx


def forward_matrix(pupil_j, k_min, k_max, instrument='lsst'):
    """The raw sensitivity matrix sliced to the same (k, j) rows the SVD uses.

    Parameters
    ----------
    pupil_j : `list` [`int`]
    k_min, k_max : `int`
        Inclusive focal (field) Zernike order range.
    instrument : `str`, optional

    Returns
    -------
    S : `numpy.ndarray`
        Shape (n_kj, 50), rows in ``kj_grid`` order, giving µm of wavefront per unit of each
        degree of freedom — µm for the rigid-body and bending-mode terms, deg for the tilts.

    Notes
    -----
    This is the *unnormalized* matrix, the same `lsst.ts.ofc` ``OFCData.sensitivity_matrix``
    that `ofc_svd.build_ofc_svd` slices before applying the normalization weights, so the
    forward and inverse numbers below come from one matrix rather than two. It is evaluated at
    camera rotator angle 0.0 deg, an AOS group decision, so both the wavefront and the DOF are
    in the Observatory Coordinate System (OCS).
    """
    S_full = np.asarray(OFCData(instrument).sensitivity_matrix)
    return S_full[int(k_min):int(k_max) + 1, np.asarray(pupil_j, dtype=int), :].reshape(
        -1, S_full.shape[-1])

<a id='data'></a>
## The SVD

Built by `ofc_svd.build_ofc_svd`, which owns the single `numpy.linalg.svd` call on the
sensitivity matrix. Nothing here re-does that decomposition.

In [4]:
pupil_j = read_pupil_j(PARAM_SET)
print(f'pupil Noll indices from {PARAM_SET}/visits.parquet:')
print(f'  {pupil_j}  (n = {len(pupil_j)}, dimensionless indices)')

svd = osv.build_ofc_svd(pupil_j, K_MIN, K_MAX, N_MODES, n_dof=N_DOF)
print(f'\nSVD: U_eff {svd.U_eff.shape}, Sigma {svd.Sigma.shape}, '
      f'n_dof {svd.n_dof}, n_keep_eff {svd.n_keep_eff}')
print(f'kj_grid: {len(svd.kj_grid)} entries = {K_MAX - K_MIN + 1} focal orders '
      f'x {len(pupil_j)} pupil Noll indices, pupil j fastest')
print(f'  first five: {svd.kj_grid[:5]}')

S = forward_matrix(pupil_j, K_MIN, K_MAX)
W_unit, KJ_IDX = unit_dz_vector(svd, K_FOCAL, J_PUPIL)
print(f'\nDZ(k={K_FOCAL}, j={J_PUPIL}) is kj_grid entry {KJ_IDX}; '
      f'forward matrix S has shape {S.shape}')

pupil Noll indices from fam_danish_1_2_0_wep17_6_1_refitWCS_bin2x/visits.parquet:
  [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26]  (n = 21, dimensionless indices)



SVD: U_eff (126, 34), Sigma (50,), n_dof 50, n_keep_eff 34
kj_grid: 126 entries = 6 focal orders x 21 pupil Noll indices, pupil j fastest
  first five: [(1, 4), (1, 5), (1, 6), (1, 7), (1, 8)]



DZ(k=1, j=4) is kj_grid entry 0; forward matrix S has shape (126, 50)


<a id='forward'></a>
## Forward: hexapod defocus to wavefront

One µm of hexapod defocus travel produces this much DZ(k=1, j=4). This direction is
unambiguous — it is a single matrix entry per axis — and it is what makes the inverse
ambiguous: the two axes produce nearly the same wavefront, differing by only a few percent.

In [5]:
dz_per_um_m2 = float(S[KJ_IDX, DOF_M2_DZ])
dz_per_um_cam = float(S[KJ_IDX, DOF_CAM_DZ])

print(f'DZ(k={K_FOCAL}, j={J_PUPIL}) produced per um of hexapod defocus travel '
      f'[um of wavefront per um of dz]:')
print(f'  M2 hexapod dz     (DOF {DOF_M2_DZ}) : {dz_per_um_m2:+.7e}')
print(f'  camera hexapod dz (DOF {DOF_CAM_DZ}) : {dz_per_um_cam:+.7e}')
print(f'  the two axes agree to '
      f'{100.0 * abs(dz_per_um_cam - dz_per_um_m2) / abs(0.5 * (dz_per_um_cam + dz_per_um_m2)):.1f}%'
      f' (dimensionless, difference over mean) and share a sign')

print(f'\nInverting each axis alone -- the constant to use when only that hexapod moves\n'
      f'[um of dz travel per um of wavefront]:')
print(f'  camera hexapod alone : {1.0 / dz_per_um_cam:+.4f}')
print(f'  M2 hexapod alone     : {1.0 / dz_per_um_m2:+.4f}')
print(f'  both, split 0.5 um each (total travel) : '
      f'{1.0 / (0.5 * (dz_per_um_cam + dz_per_um_m2)):+.4f}')

DZ(k=1, j=4) produced per um of hexapod defocus travel [um of wavefront per um of dz]:
  M2 hexapod dz     (DOF 0) : -1.5341062e-02
  camera hexapod dz (DOF 5) : -1.5913716e-02
  the two axes agree to 3.7% (dimensionless, difference over mean) and share a sign

Inverting each axis alone -- the constant to use when only that hexapod moves
[um of dz travel per um of wavefront]:
  camera hexapod alone : -62.8389
  M2 hexapod alone     : -65.1845
  both, split 0.5 um each (total travel) : -63.9902


<a id='inverse'></a>
## Inverse: DZ(k=1, j=4) to v-modes and DOF

The SVD's minimum-norm solution. This is the route `fam_dz` and `optical_state` both take —
`project_amplitudes` then `vmodes` / `dof` — so these are the constants that make a FAM DZ fit
and a corner-sensor optical state comparable.

Because the pseudo-inverse minimizes the norm of the DOF vector, it splits the defocus across
*both* hexapods rather than putting it all on one. That is a different convention from either
single-axis inverse above, not a correction to it.

In [6]:
A_modes = svd.project_amplitudes(W_unit)
vmodes = svd.vmodes(A_modes)[0]
dof = svd.dof(A_modes)[0]

v1_per_um_dz_coeff = float(vmodes[0])
print(f'Recovered from 1 um of wavefront in DZ(k={K_FOCAL}, j={J_PUPIL}):\n')
print(f'  v-mode 1 amplitude : {v1_per_um_dz_coeff:+.7e} '
      f'(dimensionless v-mode amplitude per um of wavefront)')
print(f'  || v || over all {len(vmodes)} retained modes : {np.linalg.norm(vmodes):.7e} '
      f'(dimensionless)')

order = np.argsort(-np.abs(vmodes))[:5]
print(f'\n  largest v-mode amplitudes [dimensionless per um of wavefront]:')
for m in order:
    print(f'    v{m + 1:<3d} {vmodes[m]:+.5e}')
print(f'  so DZ(k={K_FOCAL}, j={J_PUPIL}) is not a pure v-mode 1: the term excites several '
      f'modes,')
print(f'  and v-mode 1 carries '
      f'{100.0 * vmodes[0] ** 2 / float(np.sum(vmodes ** 2)):.1f}% of the total v-mode '
      f'*power* (dimensionless,')
print(f'  squared v1 amplitude over summed squared amplitudes -- power, not amplitude).')

Recovered from 1 um of wavefront in DZ(k=1, j=4):

  v-mode 1 amplitude : -5.7375710e-02 (dimensionless v-mode amplitude per um of wavefront)
  || v || over all 34 retained modes : 9.3805361e-02 (dimensionless)

  largest v-mode amplitudes [dimensionless per um of wavefront]:
    v1   -5.73757e-02
    v16  +5.52212e-02
    v21  +4.03487e-02
    v33  -2.72015e-02
    v34  -8.74998e-03
  so DZ(k=1, j=4) is not a pure v-mode 1: the term excites several modes,
  and v-mode 1 carries 37.4% of the total v-mode *power* (dimensionless,
  squared v1 amplitude over summed squared amplitudes -- power, not amplitude).


In [7]:
labels, units = svd.dof_labels()
print(f'  degrees of freedom recovered from 1 um of wavefront in '
      f'DZ(k={K_FOCAL}, j={J_PUPIL}):\n')
dof_order = np.argsort(-np.abs(dof))[:6]
for d in dof_order:
    print(f'    DOF {d:<3d} {labels[d]:<28s} {dof[d]:+12.5f} {units[d]} per um of wavefront')
print(f'\n  || dof || = {np.linalg.norm(dof):.4f} (mixed um and deg, so a scale only)')
print(f'  the two hexapod dz axes dominate: the next-largest term is '
      f'{abs(dof[dof_order[2]]):.4f},')
print(f'  a factor {abs(dof[dof_order[1]]) / abs(dof[dof_order[2]]):.0f} below the smaller of '
      f'the two (dimensionless ratio of')
print(f'  DOF magnitudes), so the minimum-norm solution is essentially pure hexapod defocus.')

cam_um = float(dof[DOF_CAM_DZ])
m2_um = float(dof[DOF_M2_DZ])
print(f'\n  equivalent hexapod defocus per um of wavefront [um of dz travel]:')
print(f'    camera hexapod dz (DOF {DOF_CAM_DZ}) : {cam_um:+.4f}')
print(f'    M2 hexapod dz     (DOF {DOF_M2_DZ}) : {m2_um:+.4f}')
print(f'    total travel (the two summed)    : {cam_um + m2_um:+.4f}')

  degrees of freedom recovered from 1 um of wavefront in DZ(k=1, j=4):

    DOF 0   M2_dz                           -35.14810 μm per um of wavefront
    DOF 5   Cam_dz                          -28.07142 μm per um of wavefront
    DOF 2   M2_dy                            +0.12900 μm per um of wavefront
    DOF 1   M2_dx                            -0.10041 μm per um of wavefront
    DOF 6   Cam_dx                           +0.09721 μm per um of wavefront
    DOF 7   Cam_dy                           -0.07980 μm per um of wavefront

  || dof || = 44.9826 (mixed um and deg, so a scale only)
  the two hexapod dz axes dominate: the next-largest term is 0.1290,
  a factor 218 below the smaller of the two (dimensionless ratio of
  DOF magnitudes), so the minimum-norm solution is essentially pure hexapod defocus.

  equivalent hexapod defocus per um of wavefront [um of dz travel]:
    camera hexapod dz (DOF 5) : -28.0714
    M2 hexapod dz     (DOF 0) : -35.1481
    total travel (the two summed) 

<a id='checks'></a>
## Cross-checks

Two things must hold. The recovered DOF must reproduce the input DZ term through the forward
matrix, and the v-mode-1 constant must agree with the one the focus studies already use.

In [8]:
# Round trip: push the recovered DOF back through the forward matrix.
back = S @ dof
leak = np.delete(back, KJ_IDX)
worst = int(np.argmax(np.abs(leak)))
kj_other = [kj for i, kj in enumerate(svd.kj_grid) if i != KJ_IDX]
print(f'Round trip through the forward matrix [um of wavefront]:')
print(f'  DZ(k={K_FOCAL}, j={J_PUPIL}) reproduced : {back[KJ_IDX]:.7f}  (input 1.0000000)')
print(f'  largest leakage into another term  : {leak[worst]:+.7f} at '
      f'(k, j) = {kj_other[worst]}')
print(f'  || reproduced - input ||           : {np.linalg.norm(back - W_unit[0]):.7f}')
print(f'\n  The leakage is the k<=6 truncation and the 34-mode cut, not an error: a unit\n'
      f'  DZ(k={K_FOCAL}, j={J_PUPIL}) is not exactly in the range of the truncated forward map.')

Round trip through the forward matrix [um of wavefront]:
  DZ(k=1, j=4) reproduced : 0.9999679  (input 1.0000000)
  largest leakage into another term  : +0.0024397 at (k, j) = (3, 7)
  || reproduced - input ||           : 0.0056666

  The leakage is the k<=6 truncation and the 34-mode cut, not an error: a unit
  DZ(k=1, j=4) is not exactly in the range of the truncated forward map.


In [9]:
# The constant the focus studies use, for comparison. It answers the *other* question --
# v-mode 1 per um of commanded hexapod travel -- so it is the reciprocal-like partner of the
# DZ constant above, not the same number.
v1_per_um_travel = A.v1_per_um_dz_value(dof_set='all_50', n_modes=N_MODES, verbose=True)

print(f'\nChaining the two constants, as a consistency check:')
print(f'  1 um of wavefront in DZ(k={K_FOCAL}, j={J_PUPIL}) -> v-mode 1 = '
      f'{v1_per_um_dz_coeff:+.7e} (dimensionless)')
print(f'  that v-mode 1, divided by v1_per_um_dz = {v1_per_um_travel:.6e} per um, gives')
print(f'    {v1_per_um_dz_coeff / v1_per_um_travel:+.4f} um of equivalent hexapod dz travel')
print(f'  against the SVD total travel of {cam_um + m2_um:+.4f} um -- they agree to '
      f'{100.0 * abs(v1_per_um_dz_coeff / v1_per_um_travel - (cam_um + m2_um)) / abs(cam_um + m2_um):.1f}%')
print(f'  (dimensionless, difference over the SVD value). The residual is the other v-modes:')
print(f'  the v-mode-1-only route discards the '
      f'{100.0 - 100.0 * vmodes[0] ** 2 / float(np.sum(vmodes ** 2)):.1f}% of v-mode power')
print(f'  DZ(k={K_FOCAL}, j={J_PUPIL}) puts into other modes, which also carry hexapod dz.')

v1 per um camera-hexapod dz (DOF 5) = -8.9144254e-04 per um
v1 per um M2-hexapod dz     (DOF 0) = -9.1026032e-04 per um
mean magnitude = 9.00851e-04 per um; axes agree to 2.1% (dimensionless)
  the two axes share a sign, so their sum over their mean is 2.00000 (dimensionless): this factor is 1 um of TOTAL dz travel, 0.5 um on each hexapod

Chaining the two constants, as a consistency check:
  1 um of wavefront in DZ(k=1, j=4) -> v-mode 1 = -5.7375710e-02 (dimensionless)
  that v-mode 1, divided by v1_per_um_dz = 9.008514e-04 per um, gives
    -63.6905 um of equivalent hexapod dz travel
  against the SVD total travel of -63.2195 um -- they agree to 0.7%
  (dimensionless, difference over the SVD value). The residual is the other v-modes:
  the v-mode-1-only route discards the 62.6% of v-mode power
  DZ(k=1, j=4) puts into other modes, which also carry hexapod dz.


<a id='results'></a>
## The constants, collected

In [10]:
rows = [
    ('forward, M2 hexapod dz alone', dz_per_um_m2,
     'um of wavefront DZ(1,4) per um of M2 dz'),
    ('forward, camera hexapod dz alone', dz_per_um_cam,
     'um of wavefront DZ(1,4) per um of camera dz'),
    ('inverse, camera hexapod alone', 1.0 / dz_per_um_cam,
     'um of camera dz per um of wavefront DZ(1,4)'),
    ('inverse, M2 hexapod alone', 1.0 / dz_per_um_m2,
     'um of M2 dz per um of wavefront DZ(1,4)'),
    ('inverse, 0.5 um on each hexapod', 1.0 / (0.5 * (dz_per_um_cam + dz_per_um_m2)),
     'um of total dz travel per um of wavefront DZ(1,4)'),
    ('SVD minimum-norm, v-mode 1', v1_per_um_dz_coeff,
     'dimensionless v-mode-1 amplitude per um of wavefront DZ(1,4)'),
    ('SVD minimum-norm, camera dz', cam_um,
     'um of camera dz per um of wavefront DZ(1,4)'),
    ('SVD minimum-norm, M2 dz', m2_um,
     'um of M2 dz per um of wavefront DZ(1,4)'),
    ('SVD minimum-norm, total dz travel', cam_um + m2_um,
     'um of total dz travel per um of wavefront DZ(1,4)'),
    ('v1_per_um_dz (the focus studies)', v1_per_um_travel,
     'dimensionless v-mode-1 amplitude per um of total dz travel'),
]
table = pd.DataFrame(rows, columns=['constant', 'value', 'units'])
print(f'DZ(k={K_FOCAL}, j={J_PUPIL}) conversion constants, focal orders '
      f'k={K_MIN}..{K_MAX}, {N_DOF} DOF, {N_MODES} v-modes,')
print(f'sensitivity matrix at camera rotator angle 0.0 deg, wavefront and DOF in OCS:\n')
with pd.option_context('display.width', 140, 'display.max_colwidth', 60):
    print(table.to_string(index=False, float_format=lambda x: f'{x:+.6e}'))

DZ(k=1, j=4) conversion constants, focal orders k=1..6, 50 DOF, 34 v-modes,
sensitivity matrix at camera rotator angle 0.0 deg, wavefront and DOF in OCS:

                         constant         value                                                        units
     forward, M2 hexapod dz alone -1.534106e-02                      um of wavefront DZ(1,4) per um of M2 dz
 forward, camera hexapod dz alone -1.591372e-02                  um of wavefront DZ(1,4) per um of camera dz
    inverse, camera hexapod alone -6.283887e+01                  um of camera dz per um of wavefront DZ(1,4)
        inverse, M2 hexapod alone -6.518454e+01                      um of M2 dz per um of wavefront DZ(1,4)
  inverse, 0.5 um on each hexapod -6.399022e+01            um of total dz travel per um of wavefront DZ(1,4)
       SVD minimum-norm, v-mode 1 -5.737571e-02 dimensionless v-mode-1 amplitude per um of wavefront DZ(1,4)
      SVD minimum-norm, camera dz -2.807142e+01                  um of camera dz p

### Which constant to use

- To convert a **FAM DZ fit into the same units as a corner-sensor optical state**, use the
  SVD minimum-norm route — or better, read `fam_dz.v_modes` directly, which is that projection
  already stored, in the same basis as `optical_state.v_modes`.
- To ask **"what camera hexapod move would produce this much DZ(1,4)?"** — the question a
  commanded Full Array Mode defocus answers, since that defocus is applied on the camera
  hexapod alone — use the camera-only inverse. It differs from the SVD total travel because the
  SVD splits the motion between both hexapods.
- The **forward** numbers are the only unambiguous ones. Every inverse embeds a choice about
  how to distribute the motion between two nearly degenerate axes, and the size of that choice
  is the gap between the rows above — not a small correction.